# PPO Optimizer Chain Demo

This is a small demo notebook for the saved PPO model. It loads five random test circuits, draws each initial circuit from its QASM file, prints the initial statistics, predicts one optimizer chain per circuit, and compares the prediction against a few baselines.


In [ ]:
from pathlib import Path
import os
import sys
import warnings

os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".mplconfig"))
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

from IPython.display import Markdown, display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from qiskit import QuantumCircuit
from sklearn.model_selection import train_test_split
from stable_baselines3 import PPO

RANDOM_STATE = 42
DEMO_RANDOM_STATE = 17
RUNTIME_WEIGHT = 0.20
MODEL_CHAIN_BASELINE = "tket__wisq_bqskit"

OPTIMIZER_NAMES = ["qiskit_standard", "qiskit_ai", "tket", "wisq_rules", "wisq_bqskit"]
METRIC_NAMES = ["num_qubits", "depth", "two_qubit_gates", "total_gates", "two_qubit_depth", "gate_density", "two_qubit_ratio",]

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "trajectories_db_analysis" / "one_and_two_opt_master.csv").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Could not find project root.")
    PROJECT_ROOT = PROJECT_ROOT.parent

CSV_PATH = PROJECT_ROOT / "trajectories_db_analysis" / "one_and_two_opt_master.csv"
CHAIN_OPTIMIZER_DIR = PROJECT_ROOT / "machine_learning" / "RL" / "chain_optimizer"
MODEL_PATH = CHAIN_OPTIMIZER_DIR / "optimizer_chain_ppo_0.2.zip"
QASM_ROOT = PROJECT_ROOT / "benchmarks" / "ai_transpile" / "qasm"

sys.path.insert(0, str(CHAIN_OPTIMIZER_DIR))
from optimizer_chain_env import OptimizerChainEnv

print(CSV_PATH)
print(MODEL_PATH)


## Load Data

This uses the same circuit-level train/validation/test split as the training notebook. The random demo circuits are sampled from the test set, so they are not in the training set.


In [ ]:
df = pd.read_csv(CSV_PATH)

df["two_qubit_gate_reduction"] = (
    df["initial_two_qubit_gates"] - df["final_two_qubit_gates"]
)
df["two_qubit_gate_reduction_pct"] = (
    100
    * df["two_qubit_gate_reduction"]
    / df["initial_two_qubit_gates"].replace(0, np.nan)
)

df = df.dropna(
    subset=[
        "two_qubit_gate_reduction_pct",
        "total_runtime_seconds",
        "final_two_qubit_gates",
    ]
).copy()

CATEGORY_NAMES = sorted(df["original_category"].unique())

circuits = np.array(sorted(df["original_circuit"].unique()))
train_val_circuits, test_circuits = train_test_split(
    circuits,
    test_size=0.10,
    random_state=RANDOM_STATE,
)
train_circuits, val_circuits = train_test_split(
    train_val_circuits,
    test_size=0.10 / 0.90,
    random_state=RANDOM_STATE,
)

train_df = df[df["original_circuit"].isin(train_circuits)].copy()
val_df = df[df["original_circuit"].isin(val_circuits)].copy()
test_df = df[df["original_circuit"].isin(test_circuits)].copy()

train_runtime_99 = float(train_df["total_runtime_seconds"].quantile(0.99))
runtime_denominator = np.log1p(max(train_runtime_99, 1e-9))

for frame in [train_df, val_df, test_df]:
    frame["pct_score"] = frame["two_qubit_gate_reduction_pct"] / 100.0
    frame["runtime_score"] = (
        np.log1p(frame["total_runtime_seconds"].clip(lower=0)) / runtime_denominator
    ).clip(0, 2)
    frame["utility_reward"] = frame["pct_score"] - RUNTIME_WEIGHT * frame["runtime_score"]

print(f"Training circuits: {len(train_circuits)}")
print(f"Validation circuits: {len(val_circuits)}")
print(f"Test circuits available for demo: {test_df['original_circuit'].nunique()}")


In [ ]:
def initial_metric_frame(frame):
    return pd.DataFrame(
        {
            "num_qubits": frame["initial_num_qubits"],
            "depth": frame["initial_depth"],
            "two_qubit_gates": frame["initial_two_qubit_gates"],
            "total_gates": frame["initial_total_gates"],
            "two_qubit_depth": frame["initial_two_qubit_depth"],
            "gate_density": frame["initial_gate_density"],
            "two_qubit_ratio": frame["initial_two_qubit_ratio"],
        }
    )


def final_metric_frame(frame):
    final_total = frame["final_total_gates"].replace(0, np.nan)
    final_qubits = frame["final_num_qubits"].replace(0, np.nan)
    return pd.DataFrame(
        {
            "num_qubits": frame["final_num_qubits"],
            "depth": frame["final_depth"],
            "two_qubit_gates": frame["final_two_qubit_gates"],
            "total_gates": frame["final_total_gates"],
            "two_qubit_depth": frame["final_two_qubit_depth"],
            "gate_density": frame["final_total_gates"] / final_qubits,
            "two_qubit_ratio": frame["final_two_qubit_gates"] / final_total,
        }
    ).fillna(0.0)


train_single_df = train_df[~train_df["chain_name"].str.contains("__", regex=False)]
scale_source = pd.concat(
    [initial_metric_frame(train_df), final_metric_frame(train_single_df)],
    ignore_index=True,
)
metric_scale = scale_source.quantile(0.95).replace(0, 1.0).to_dict()
metric_scale


## Pick Five Random Test Circuits


In [ ]:
eligible_circuits = []
for circuit, rows in test_df.groupby("original_circuit"):
    chains = set(rows["chain_name"])
    if MODEL_CHAIN_BASELINE in chains and any("__" not in chain for chain in chains):
        eligible_circuits.append(circuit)

rng = np.random.default_rng(DEMO_RANDOM_STATE)
selected_circuits = sorted(rng.choice(eligible_circuits, size=5, replace=False))
selected_circuits


## Visualize the Initial Circuits and Initial Stats


In [ ]:
qasm_files = {}
for path in QASM_ROOT.rglob("*.qasm"):
    qasm_files.setdefault(path.stem, []).append(path)

for circuit in selected_circuits:
    row = test_df[test_df["original_circuit"] == circuit].iloc[0]
    display(Markdown(f"### {circuit} ({row['original_category']})"))

    stats = pd.DataFrame(
        {
            "metric": [
                "num_qubits",
                "depth",
                "two_qubit_gates",
                "total_gates",
                "two_qubit_depth",
                "gate_density",
                "two_qubit_ratio",
            ],
            "value": [
                row["initial_num_qubits"],
                row["initial_depth"],
                row["initial_two_qubit_gates"],
                row["initial_total_gates"],
                row["initial_two_qubit_depth"],
                row["initial_gate_density"],
                row["initial_two_qubit_ratio"],
            ],
        }
    )
    display(stats)

    qasm_path = Path(str(row["original_circuit_path"]))
    if not qasm_path.exists():
        names_to_try = [circuit]
        if circuit.startswith("guoq_"):
            names_to_try.append(circuit.removeprefix("guoq_"))
        qasm_path = None
        for name in names_to_try:
            if name in qasm_files:
                candidates = sorted(qasm_files[name])
                category = str(row["original_category"])
                category_matches = [p for p in candidates if category in str(p)]
                qasm_path = category_matches[0] if category_matches else candidates[0]
                break

    if qasm_path is None:
        print("Could not find local QASM file.")
        continue

    print(qasm_path)
    qc = QuantumCircuit.from_qasm_file(str(qasm_path))

    try:
        fig = qc.draw(output="mpl", fold=35, idle_wires=False)
        display(fig)
        plt.close(fig)
    except Exception:
        print(qc.draw(output="text", fold=120))


## Load the Model and Predict One Chain at a Time

This uses the same `OptimizerChainEnv` class that the PPO model was trained with, imported from `optimizer_chain_env.py`.


In [ ]:
prediction_env = OptimizerChainEnv(
    test_df,
    OPTIMIZER_NAMES,
    CATEGORY_NAMES,
    metric_scale,
    shuffle_each_epoch=False,
)
model = PPO.load(MODEL_PATH, env=prediction_env)


    

# Test One Circuit

In [ ]:
first_circuit = selected_circuits[0]

optimizer_chain = ""

obs, _ = prediction_env.reset(options={"circuit": first_circuit})
while True:
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = prediction_env.step(int(action))
    
    if terminated:
        optimizer_chain = info["selected_chain"]
        break

print(f"Circuit Name: {first_circuit}")
print(f"Optimizer Chain: {optimizer_chain}")
init_row = test_df[test_df["original_circuit"] == circuit].iloc[0]
print(f"Initial 2Q Gates: {int(init_row.initial_two_qubit_gates)}")
opt_row = test_df[(test_df["original_circuit"] == circuit) & (test_df["chain_name"] == optimizer_chain)]
print(f"Final 2Q Gates: {int(opt_row.final_two_qubit_gates)}")

In [ ]:
predictions = []

for circuit in selected_circuits:
    obs, _ = prediction_env.reset(options={"circuit": circuit})

    action, _ = model.predict(obs, deterministic=True)
    actions = [int(action)]
    obs, reward, terminated, truncated, info = prediction_env.step(int(action))

    if not (terminated or truncated):
        action, _ = model.predict(obs, deterministic=True)
        actions.append(int(action))
        obs, reward, terminated, truncated, info = prediction_env.step(int(action))

    predicted_chain = info["selected_chain"]

    print(f"{circuit}: {predicted_chain}")

    predictions.append(
        {
            "circuit": circuit,
            "ppo_chain": predicted_chain,
            "actions": actions,
            "invalid_action": info.get("invalid_action", False),
        }
    )

prediction_df = pd.DataFrame(predictions)
prediction_df


## Compare PPO to Baselines

Baselines:

- `tket__wisq_bqskit`
- best single optimizer for that specific circuit
- oracle best chain for that specific circuit


In [ ]:
comparison_rows = []

for item in predictions:
    circuit = item["circuit"]
    rows = test_df[test_df["original_circuit"] == circuit]

    ppo_row = rows[rows["chain_name"] == item["ppo_chain"]]
    fixed_row = rows[rows["chain_name"] == MODEL_CHAIN_BASELINE]
    single_rows = rows[~rows["chain_name"].str.contains("__", regex=False)]
    oracle_row = rows.loc[[rows["utility_reward"].idxmax()]]

    methods = [
        ("PPO", ppo_row),
        ("tket__wisq_bqskit", fixed_row),
        ("oracle", oracle_row),
    ]

    for method, method_rows in methods:

        row = method_rows.iloc[0]
        comparison_rows.append(
            {
                "circuit": circuit,
                "method": method,
                "chain": row["chain_name"],
                "utility_reward": row["utility_reward"],
                "total_runtime_seconds": row["total_runtime_seconds"],
                "two_qubit_gate_reduction": row["two_qubit_gate_reduction"],
                "two_qubit_gate_reduction_pct": row["two_qubit_gate_reduction_pct"],
            }
        )

comparison_df = pd.DataFrame(comparison_rows)
comparison_df


In [ ]:
plot_df = comparison_df.copy()
method_order = ["PPO", "tket__wisq_bqskit",  "oracle"]
plot_df["method"] = pd.Categorical(plot_df["method"], categories=method_order, ordered=True)

utility_table = plot_df.pivot(index="circuit", columns="method", values="utility_reward")
ax = utility_table[method_order].plot(kind="bar", figsize=(12, 5))
ax.set_title("Utility reward by circuit")
ax.set_ylabel("utility reward")
ax.set_xlabel("circuit")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

runtime_table = plot_df.pivot(index="circuit", columns="method", values="total_runtime_seconds")
runtime_table[method_order].plot(kind="bar", ax=axes[0])
axes[0].set_title("Runtime by circuit")
axes[0].set_ylabel("seconds")
axes[0].set_xlabel("circuit")
axes[0].tick_params(axis="x", rotation=45)

reduction_table = plot_df.pivot(index="circuit", columns="method", values="two_qubit_gate_reduction")
reduction_table[method_order].plot(kind="bar", ax=axes[1])
axes[1].set_title("Two-qubit gate reduction by circuit")
axes[1].set_ylabel("gate reduction")
axes[1].set_xlabel("circuit")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()
